# Warsztat 3: YOLO jako czujnik w minigrze

### Cel ćwiczenia

Gracz pokazuje przedmiot przed kamerą. Jeśli **butelka** znajdzie się w oznaczonej strefie obrazu, gra przyznaje **nagrodę**. Oczy gry to gotowa sieć neuronowa YOLO, ale decyzję o nagrodzie podejmuje napisana przez Was reguła.

W podstawowym wariancie zamiast kamery używamy 12 przygotowanych zdjęć, dzięki czemu każdy zespół testuje dokładnie to samo. Waszym zadaniem jest połączyć wynik sieci z regułą gry i sprawdzić, jak **próg ufności** wpływa na błędne decyzje. Modelu nie trenujemy.

### Dane wejściowe

Zdjęcia leżą w folderze `data/images`, a plik `data/labels.csv` opisuje każde z nich:

```markdown
  Kolumna  |      Znaczenie
-----------|-------------------------------------------
   file    |   Nazwa pliku
   group   |   w_strefie, poza_strefa albo bez_butelki
   label   |   nagroda albo brak nagrody
```

Po cztery zdjęcia w każdej grupie. Kolumna `label` to **etykieta referencyjna**, czyli poprawna odpowiedź, z którą porównujemy decyzję gry. Program używa jej tylko do oceny, nigdy do podjęcia decyzji.

---
Model zwraca dla każdego wykrytego obiektu trzy informacje:

| Pojęcie | Co oznacza | Przykład |
|---|---|---|
| **klasa** | co to jest, według listy znanej modelowi | `bottle` |
| **ramka ograniczająca** | gdzie to jest: współrzędne lewego górnego i prawego dolnego rogu w pikselach | `x1=120, y1=40, x2=210, y2=300` |
| **wynik ufności** | jak bardzo model jest pewny tej detekcji, od 0 do 1 | `0.83` |

Ufność nie jest prawdopodobieństwem poprawności. To sygnał, który porównujemy z progiem wybranym przez nas.

### 1. Przygotowanie

Importujemy biblioteki i funkcje pomocnicze z `yolo_utils.py` oraz ustawiamy parametry eksperymentu.

In [1]:
import matplotlib.pyplot as plt
from ultralytics import YOLO
from yolo_utils import (
    CORRECT_REFUSAL, CORRECT_REWARD, MISSED_REWARD, OUTCOMES, REWARD, TARGET_CLASS, WRONG_REWARD,
    Detection, TestRow, detect, draw_decision, draw_detections, load_samples, print_detections,
    print_summary, print_test_table, read_image, show_image, show_samples,
)

MODEL_PATH = "yolo11n.pt"
LABELS_PATH = "data/labels.csv"
THRESHOLDS = [0.25, 0.60]

### 2. Dane

Wczytujemy opis zdjęć i oglądamy je. Wybierzcie dwa zdjęcia i zanim uruchomicie model, zapiszcie w sprawozdaniu, jaką decyzję powinna podjąć gra.

In [2]:
# TODO: Wczytaj próbki korzystając z funkcji load_samples i ścieżki LABELS_PATH.
samples = ...

# TODO: Wyświetl nazwę pliku (name), grupę (group) i etykietę (label) każdej próbki.
print(f"{'PLIK':<12}{'GRUPA':<14}{'ETYKIETA'}")
for sample in samples:
    print(f"{...:<12}{...:<14}{...}")

show_samples(samples)

PLIK        GRUPA         ETYKIETA


TypeError: 'ellipsis' object is not iterable

### 3. Model

Korzystamy z gotowego, małego modelu `yolo11n.pt` wytrenowanego na zbiorze COCO, który obejmuje 80 klas. Plik z wagami jest w pakiecie, więc nic nie trzeba pobierać. Model zna tylko te klasy, dlatego nie zakładamy, że rozpozna dowolny przedmiot z gry.

W `model.names` znajdziecie słownik: numer klasy → nazwa klasy.

In [ ]:
# TODO: Wczytaj model klasą YOLO, podając ścieżkę MODEL_PATH.
model = ...

# TODO: Wypisz liczbę klas modelu (długość słownika model.names).
print(f"Liczba klas: {...}")

# TODO: Sprawdź, czy TARGET_CLASS jest wśród wartości słownika model.names.
print(f"Czy model zna klasę '{TARGET_CLASS}'? {...}")

### 4. Detekcja na jednym zdjęciu

Gotowa funkcja `detect(model, image)` uruchamia model i zwraca obiekt `result` klasy `ModelResult`. To uproszczona wersja obiektu `Results` z biblioteki ultralytics: zostawiliśmy w nim tylko to, czego potrzebujecie, zapisane zwykłymi typami Pythona.

**Definicje klas `ModelResult` i `Box` znajdziecie w pliku `yolo_utils.py`.** Gdy nie wiecie, co zawiera `result`, zajrzyjcie właśnie tam.

| Co odczytujemy | Kod | Typ | Przykład |
|---|---|---|---|
| wszystkie wykryte ramki | `result.boxes` | lista obiektów `Box` | `[]`, gdy model nic nie znalazł |
| numer klasy ramki | `box.class_id` | `int` | `39` |
| nazwa klasy o danym numerze | `result.names[numer_klasy]` | `str` | `result.names[39]` to `"bottle"` |
| ufność ramki | `box.confidence` | `float` | `0.85` |
| współrzędne ramki | `box.xyxy` | 4 liczby `(x1, y1, x2, y2)` | `(425.4, 263.5, 481.0, 375.0)` |

Ramka zna tylko **numer** klasy. Nazwę podaje słownik `result.names`, ten sam co `model.names` z sekcji 3.

Tak wygląda `result` dla zdjęcia `img02.jpg` (wartości zaokrąglone):

```python
ModelResult(
    names={0: 'person', 1: 'bicycle', ..., 39: 'bottle', ..., 79: 'toothbrush'},  # 80 klas
    boxes=[
        Box(class_id=71, confidence=0.89, xyxy=(170.7, 196.1, 321.0, 266.7)),  # 71 to 'sink'
        Box(class_id=39, confidence=0.85, xyxy=(425.4, 263.5, 481.0, 375.0)),  # 39 to 'bottle'
        Box(class_id=39, confidence=0.83, xyxy=(215.3, 110.6, 239.0, 191.1)),  # 39 to 'bottle'
    ],
)
```

Uzupełnij funkcję `read_detections`, która zamienia `result` na listę prostych obiektów `Detection`, w których zamiast numeru klasy jest już jej nazwa.

Funkcja `detect` używa niskiego progu `0.10`, żeby widzieć także słabe detekcje. Progi gry nałożymy dopiero w kodzie, co pozwoli później odróżnić brak detekcji od detekcji poniżej progu.

In [ ]:
def read_detections(result):
    # result to obiekt ModelResult, opisany w yolo_utils.py.
    detections = []

    for box in result.boxes:
        # TODO: Odczytaj nazwę klasy: numer klasy ramki zamień na nazwę słownikiem result.names.
        class_name = ...

        # TODO: Odczytaj ufność ramki.
        confidence = ...

        # TODO: Odczytaj współrzędne ramki.
        x1, y1, x2, y2 = ...

        detections.append(Detection(class_name, confidence, x1, y1, x2, y2))

    return detections

In [ ]:
sample = samples[1]
image = read_image(sample.path)

# TODO: Uruchom detect na obrazie, a wynik zamień przez read_detections.
detections = ...

print_detections(detections)
show_image(draw_detections(image, detections), sample.name)
plt.show()

### 5. Strefa

Strefa to **środkowa jedna trzecia szerokości obrazu, na całej jego wysokości**. Dla obrazu o szerokości 600 pikseli strefa zaczyna się na `x = 200` i kończy na `x = 400`. Zdjęcia mają różne rozmiary, dlatego liczymy ją osobno dla każdego.

Skąd wziąć szerokość? Funkcja `read_image` zwraca obraz jako tablicę NumPy, czyli siatkę pikseli. Jej rozmiar podaje atrybut `image.shape`: trzy liczby `(wysokość, szerokość, kanały)`. Na przykład dla `img02.jpg` `image.shape` to `(375, 500, 3)`: 375 wierszy pikseli, 500 kolumn i 3 kanały koloru.

| Co | Kod | Dla `img02.jpg` |
|---|---|---|
| wysokość obrazu | `image.shape[0]` | `375` |
| szerokość obrazu | `image.shape[1]` | `500` |

In [ ]:
def get_zone(image_width):
    # TODO: Oblicz lewą i prawą granicę strefy.
    left = ...
    right = ...

    return left, right

In [ ]:
# Sprawdzenie: ta komórka nie może zgłosić błędu.
assert get_zone(600) == (200, 400)

# TODO: Oblicz strefę dla bieżącego obrazu.
zone = ...
print(f"Wymiary obrazu (wysokość, szerokość, kanały): {image.shape}")
print(f"Szerokość obrazu: {image.shape[1]}, strefa: od {zone[0]:.0f} do {zone[1]:.0f}")

### 6. Reguła gry

**Nagroda przysługuje, gdy co najmniej jedna wykryta butelka spełnia oba warunki:**

1. jej ufność przekracza próg,
2. środek jej ramki znajduje się w strefie.

Jedno zdjęcie daje najwyżej jedną nagrodę, nawet gdy warunki spełni kilka butelek. 

Liczy się **środek** ramki, a nie to, czy ramka zahacza o strefę.

Środek w poziomie to średnia z `x1` i `x2`.

In [ ]:
def center_x(detection):
    # TODO: Oblicz współrzędną x środka ramki.
    return ...


def in_zone(x, zone):
    # TODO: Sprawdź, czy x leży między granicami strefy (granice należą do strefy).
    return ...


def reward_decision(detections, zone, threshold):
    for detection in detections:
        # TODO: Uzupełnij trzy warunki: klasa to TARGET_CLASS, ufność przekracza próg, środek ramki jest w strefie.
        if ... and ... and ...:
            return True

    return False

In [ ]:
# Sprawdzenie na ręcznie przygotowanych ramkach: ta komórka nie może zgłosić błędu.
test_zone = (200, 400)
in_zone_box = Detection("bottle", 0.80, 280, 50, 320, 200)      # środek x = 300, w strefie
edge_box = Detection("bottle", 0.80, 390, 50, 500, 200)         # zahacza o strefę, ale środek x = 445
weak_box = Detection("bottle", 0.20, 280, 50, 320, 200)         # w strefie, ale słaba
cup_box = Detection("cup", 0.90, 280, 50, 320, 200)             # w strefie, ale to nie butelka

assert center_x(in_zone_box) == 300
assert in_zone(300, test_zone) and in_zone(200, test_zone) and not in_zone(445, test_zone)
assert reward_decision([in_zone_box], test_zone, 0.25) is True
assert reward_decision([edge_box], test_zone, 0.25) is False
assert reward_decision([weak_box], test_zone, 0.25) is False
assert reward_decision([cup_box], test_zone, 0.25) is False
assert reward_decision([cup_box, edge_box, in_zone_box], test_zone, 0.25) is True
assert reward_decision([], test_zone, 0.25) is False
print("Reguła gry działa na przypadkach testowych.")

#### Decyzja na zdjęciu

Funkcja `draw_decision` rysuje strefę, wszystkie butelki z punktem w środku ramki oraz decyzję gry. Zielona ramka przekracza próg, pomarańczowa nie.

In [ ]:
threshold = THRESHOLDS[0]

# TODO: Podejmij decyzję dla bieżącego obrazu.
reward = ...

show_image(draw_decision(image, detections, zone, threshold, reward), sample.name)
plt.show()

### 7. Test wszystkich zdjęć dla dwóch progów

Najpierw jednorazowo liczymy detekcje i strefy dla 12 zdjęć. Dzięki temu oba progi oceniamy na **tych samych** wynikach modelu.

In [ ]:
all_detections = {}
zones = {}

for sample in samples:
    image = read_image(sample.path)

    # TODO: Zapisz detekcje tego zdjęcia pod kluczem sample.name.
    all_detections[sample.name] = ...

    # Strefa liczona jak w sekcji 5: image.shape[1] to szerokość obrazu.
    zones[sample.name] = get_zone(image.shape[1])

print(f"Policzono detekcje dla {len(all_detections)} zdjęć.")

Porównujemy decyzję gry z etykietą referencyjną. Możliwe są cztery wyniki:

| | gra dała nagrodę | gra odmówiła |
|---|---|---|
| **etykieta: nagroda** | poprawna nagroda (`CORRECT_REWARD`) | pominięta nagroda (`MISSED_REWARD`) |
| **etykieta: brak nagrody** | błędna nagroda (`WRONG_REWARD`) | poprawna odmowa (`CORRECT_REFUSAL`) |

In [ ]:
def classify(label, reward):
    # TODO: Zwróć jeden z czterech wyników zgodnie z tabelą powyżej.
    if label == REWARD:
        return ...
    return ...


def run_test(threshold):
    rows = []

    for sample in samples:
        # TODO: Podejmij decyzję dla tego zdjęcia przy zadanym progu.
        reward = ...

        # TODO: Sklasyfikuj wynik na podstawie etykiety i decyzji.
        outcome = ...

        rows.append(TestRow(sample.name, sample.group, sample.label, reward, outcome))

    return rows


def count_outcomes(rows):
    counts = {outcome: 0 for outcome in OUTCOMES}

    for row in rows:
        counts[row.outcome] += 1

    return counts

In [ ]:
results = {}
counts = {}

for threshold in THRESHOLDS:
    # TODO: Uruchom test dla tego progu i policz jego wyniki.
    results[threshold] = ...
    counts[threshold] = ...

    print_test_table(results[threshold], all_detections, threshold)

print_summary(counts)

#### Skąd biorą się pomyłki?

Wyświetlamy wszystkie zdjęcia, które przy którymkolwiek progu dały błędną lub pominiętą nagrodę. Dla każdej pomyłki ustalcie przyczynę i wpiszcie ją do sprawozdania:

- **brak detekcji**: modelu w ogóle nie widzi butelki albo widzi ją poniżej progu,
- **niepoprawna lokalizacja lub klasa**: model pomylił przedmiot z butelką albo ramka jest w złym miejscu,
- **sama reguła gry**: model zadziałał poprawnie, a decyzja i tak jest niezgodna z etykietą.

In [ ]:
samples_by_name = {sample.name: sample for sample in samples}

# TODO: Zbierz nazwy zdjęć, których wynik to WRONG_REWARD albo MISSED_REWARD, przy dowolnym progu.
error_names = sorted({... for rows in results.values() for row in rows if ...})
print("Zdjęcia z pomyłkami:", error_names)

fig, axes = plt.subplots(len(error_names), len(THRESHOLDS), figsize=(11, 3.8 * len(error_names)), squeeze=False)
for row_axes, name in zip(axes, error_names):
    image = read_image(samples_by_name[name].path)
    for ax, threshold in zip(row_axes, THRESHOLDS):
        reward = reward_decision(all_detections[name], zones[name], threshold)
        canvas = draw_decision(image, all_detections[name], zones[name], threshold, reward)
        show_image(canvas, f"{name}, próg {threshold}", ax=ax)
plt.tight_layout()
plt.show()

#### Zapis przykładów do sprawozdania

Zapisujemy po jednym zdjęciu poprawnym i problematycznym. Wybierzcie własne.

In [ ]:
# TODO: Wpisz nazwę jednego zdjęcia z poprawną decyzją i jednego z pomyłką (zob. error_names).
good_name = ...
bad_name = ...

for prefix, name in [("przyklad_poprawny", good_name), ("przyklad_problem", bad_name)]:
    image = read_image(samples_by_name[name].path)
    threshold = THRESHOLDS[0]
    reward = reward_decision(all_detections[name], zones[name], threshold)
    canvas = draw_decision(image, all_detections[name], zones[name], threshold, reward)
    show_image(canvas, f"{prefix}: {name}, próg {threshold}")
    plt.savefig(f"{prefix}.png", dpi=120, bbox_inches="tight")
    plt.show()

### 8. Wnioski

Uzupełnij sprawozdanie wynikami swoich uruchomień. Odpowiedz:

1. Ile poprawnych, błędnych i pominiętych nagród oraz poprawnych odmów dał każdy próg? Który próg wybralibyście do tej minigry i dlaczego?
2. Jaka jest przyczyna każdej pomyłki: brak detekcji, niepoprawna lokalizacja lub klasa czy sama reguła gry?
3. Co kosztuje grę błędne przyznanie nagrody, a co pominięcie poprawnej akcji gracza?
4. Co dzieje się na zdjęciu, na którym ramka zahacza o strefę, ale jej środek leży poza nią? Czy taka reguła jest uczciwa dla gracza?
5. Dlaczego 12 zdjęć nie wystarcza do ogólnej oceny jakości detektora?
6. Podaj nazwę modelu, wersje bibliotek oraz wkład każdej osoby.

Oddaj również częściowe rozwiązanie, opisując jego stan. Nie wklejaj całego kodu do sprawozdania. Materiały prześlij do końca 90 minut. Rozszerzenie poniżej nie jest wymagane do oddania podstawy.

### 9. Dla chętnych: kamera

Rozszerzenie nie wymaga notatnika. Otwórz plik `kamera.py`, uzupełnij TODO i uruchom go w terminalu.

Żeby jeden nieruchomy przedmiot nie zdobywał punktu w każdej klatce, gra działa tak:

1. Punkt przyznajemy dopiero po **trzech kolejnych** klatkach z butelką w strefie.
2. Kolejny punkt odblokowujemy dopiero po **opuszczeniu strefy**, czyli po trzech kolejnych klatkach bez butelki w strefie.

Logikę sprawdzisz bez kamery:

```bash
python kamera.py --test
```

A grę uruchomisz tak (klawisz `q` kończy program):

```bash
python kamera.py --threshold 0.6
```

Sprawdźcie, jak zachowuje się licznik przy obu progach. Zapiszcie w sprawozdaniu jedną obserwację.